# 03. Colab Evaluation Pipeline

이 노트북은 정시원 담당 평가 파이프라인을 Colab에서 실행하기 위한 실행용 노트북입니다.

실행 흐름:

1. Google Drive mount
2. 프로젝트 경로 설정
3. 필요한 패키지 설치
4. 입력/출력 경로 확인
5. `pipeline/run_pipeline.py` 실행
6. `final_metrics.csv`, `final_metrics.md`, `demo_results.json` 확인

주의: `data/sample_200/annotations_sample_200.json`이 없으면 `mAP`, `AP50`, `AP75`는 `NA`로 생성됩니다.

## 1. Google Drive Mount

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Project Path 설정

프로젝트 폴더가 다른 위치에 있으면 `PROJECT_DIR`만 수정하세요.

In [ ]:
from pathlib import Path
import os

PROJECT_DIR = Path('/content/drive/MyDrive/night-detection-project')
os.chdir(PROJECT_DIR)

print('Current working directory:')
print(Path.cwd())
print('\nTop-level files:')
for path in sorted(PROJECT_DIR.iterdir()):
    print('-', path.name)

## 3. 필요한 패키지 설치

- `pyyaml`: `config.yaml` 읽기
- `pycocotools`: COCO mAP/AP50/AP75 계산

annotation 파일이 아직 없더라도 설치해두면 나중에 바로 평가할 수 있습니다.

In [ ]:
!pip install -q pyyaml pycocotools

## 4. 공통 경로 확인

In [ ]:
import yaml

CONFIG_PATH = PROJECT_DIR / 'pipeline' / 'config.yaml'

with open(CONFIG_PATH, 'r', encoding='utf-8') as f:
    config = yaml.safe_load(f)

print('Config path:', CONFIG_PATH)
print('\nAnnotation:')
print('-', config['annotation'])

print('\nConditions:')
for condition, spec in config['conditions'].items():
    print(f'[{condition}]')
    for key in ['image_dir', 'pred_json', 'vis_dir']:
        path = PROJECT_DIR / spec[key]
        status = 'OK' if path.exists() else 'MISSING'
        print(f'  {key}: {spec[key]} ({status})')

print('\nOutputs:')
for key, value in config['outputs'].items():
    print(f'- {key}: {value}')

## 5. 입력 파일 상태 점검

`pred_*.json`은 있어야 평균 detection/confidence를 계산할 수 있습니다. `annotations_sample_200.json`이 있으면 COCO mAP까지 계산됩니다.

In [ ]:
required_paths = [
    config['annotation'],
    *[spec['pred_json'] for spec in config['conditions'].values()],
]

for rel_path in required_paths:
    path = PROJECT_DIR / rel_path
    status = 'OK' if path.exists() else 'MISSING'
    print(f'{status:8} {rel_path}')

annotation_path = PROJECT_DIR / config['annotation']
if not annotation_path.exists():
    print('\n주의: annotation 파일이 없어 mAP/AP50/AP75는 NA로 생성됩니다.')

## 6. Pipeline 실행

아래 셀은 `final_metrics.csv`, `final_metrics.md`, `demo_results.json`을 생성합니다.

In [ ]:
!python pipeline/run_pipeline.py --config pipeline/config.yaml

## 7. final_metrics.csv 확인

In [ ]:
import pandas as pd

metrics_csv = PROJECT_DIR / config['outputs']['metrics_csv']
metrics_df = pd.read_csv(metrics_csv)
display(metrics_df)

## 8. final_metrics.md 확인

In [ ]:
metrics_md = PROJECT_DIR / config['outputs']['metrics_md']
print(metrics_md.read_text(encoding='utf-8'))

## 9. demo_results.json 확인

In [ ]:
import json

demo_json = PROJECT_DIR / config['outputs']['demo_json']
with open(demo_json, 'r', encoding='utf-8') as f:
    demo_results = json.load(f)

print('Demo item count:', len(demo_results))
print('\nFirst item:')
print(json.dumps(demo_results[0], indent=2, ensure_ascii=False))

## 10. 다음에 확인할 것

- `data/sample_200/annotations_sample_200.json` 추가 후 mAP/AP50/AP75 재계산
- `data/sample_200/original/` 이미지 폴더 추가
- `data/sample_200/low_light_gamma3.0/` 이미지 폴더 추가
- 데모 웹에서 `results/demo/demo_results.json` 경로를 읽도록 연결